In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller

plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# Section 1: Bitcoin Price Series and Log Returns

btc = pd.read_csv("bitcoin_data.csv", index_col="Date", parse_dates=True)
btc["Log Return"] = np.log(btc["Adj Close"] / btc["Adj Close"].shift(1))
log_returns = btc["Log Return"].dropna()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(btc.index, btc["Adj Close"], color="blue", linewidth=0.8)
axes[0].set_title("Bitcoin Price Series (Adj Close)")
axes[0].set_ylabel("Price (USD)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(log_returns.index, log_returns, color="orange", linewidth=0.6)
axes[1].set_title("Bitcoin Log Returns")
axes[1].set_ylabel("Log Return")
axes[1].axhline(0, color="black", linewidth=0.5, linestyle="--")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Section 2: ACF of Log Returns, Absolute Returns, and Squared Returns

fig, axes = plt.subplots(3, 1, figsize=(12, 12))

plot_acf(log_returns, lags=50, ax=axes[0], title="ACF of Log Returns")
plot_acf(np.abs(log_returns), lags=50, ax=axes[1], title="ACF of Absolute Log Returns")
plot_acf(log_returns**2, lags=50, ax=axes[2], title="ACF of Squared Log Returns")

for ax in axes:
    ax.set_xlabel("Lag")
    ax.set_ylabel("Autocorrelation")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Section 3: Augmented Dickey-Fuller (ADF) Test
#
# Null hypothesis (H0): the series has a unit root (non-stationary)
# Alternative hypothesis (H1): the series is stationary
# Reject H0 when ADF statistic < critical value, or p-value < 0.05

def run_adf(series, name):
    result = adfuller(series.dropna())
    print(f"--- ADF Test for {name} ---")
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4e}")
    print("Critical Values:")
    for key, val in result[4].items():
        print(f"        {key}: {val:.4f}")
    if result[1] < 0.05:
        print(f"Result: Reject the Null Hypothesis. The {name} is stationary.")
    else:
        print(f"Result: Fail to Reject the Null Hypothesis. The {name} is non-stationary.")
    print()

run_adf(btc["Adj Close"], "Price Series")
run_adf(log_returns, "Log Return Series")

--- ADF Test for Price Series ---
ADF Statistic: -0.3379
p-value: 9.1997e-01
Critical Values:
        1%: -3.4329
        5%: -2.8627
        10%: -2.5674
Result: Fail to Reject the Null Hypothesis. The Price Series is non-stationary.

--- ADF Test for Log Return Series ---
ADF Statistic: -34.9058
p-value: 0.0000e+00
Critical Values:
        1%: -3.4329
        5%: -2.8627
        10%: -2.5674
Result: Reject the Null Hypothesis. The Log Return Series is stationary.
